# HydraY NNUE - 8 king bucket, mappa a refinement stretto

Runtime -> **GPU (T4)**, poi esegui le celle in ordine. ~4h in **otto tappe**.

### La domanda
La mappa dei king bucket oggi ha 4 celle e mette **le traverse 3-8 in una sola**:
un re attivo in settima condivide i pesi con un re in terza. Qui diventano 8.

**Cambia SOLO la mappa.** Stesso dataset v7, stesso budget 160 superbatch,
stesse quattro fette nello stesso ordine, stessa architettura 1024 -> 16 -> 1,
stesso WDL, stesso learning rate. Questo notebook e' una copia di
`colab_deep_160sb.ipynb`, quello con cui e' stata addestrata la rete adottata,
con cambiati solo il branch e i controlli di taglia.

### Perche' riaprire una domanda gia' chiusa due volte
`A4-bis CHIUSA DEFINITIVAMENTE 2026-08-01: 4 bucket`, su due SPRT concordi,
**-12,37 +-9,34** e **-10,43 +-8,41**. Ma entrambi giravano a **40 superbatch**,
e sei giorni dopo abbiamo misurato che 80 contro 40 valgono **+29,4**: gli 8
bucket hanno il doppio dei parametri di l0, quindi a budget uguale ricevono
meta' addestramento per parametro. E' lo stesso argomento che questo repo
applica gia' a HalfKA contro 768, e li' non era stato applicato.

### Cosa e' diverso rispetto al tentativo del 2026-08-25
Quel tentativo era pronto ma non e' mai stato eseguito, e aveva due difetti:

1. **addestrava l'architettura sbagliata.** Il notebook chiamava `trainer`, non
   `trainer_deep`: avrebbe prodotto una rete a un layer, che il motore non
   carica piu'. E `trainer_deep.rs`/`sanity_deep.rs` erano rimasti a 4 bucket.
2. **la mappa tagliava fra le case di arrocco.** Era `0,1,2,3` sulla traversa 1,
   quindi g1 e h1 in bucket con pesi indipendenti. Il post-mortem del 2026-08-01
   aveva lasciato proprio quello come spiegazione superstite della sconfitta
   ("case di re adiacenti in bucket separati, e la mappa suddivideva la traversa
   1 dove il re sta quasi sempre") e diceva che un secondo tentativo doveva
   spostare i confini LONTANO dalle case di arrocco.

Su 16M di lookup campionati da v7 la distribuzione dice quanto pesa:

| cella | quota |
|---|---|
| traversa 1, b/g (il re arroccato) | **23,0%** |
| traversa 1, d/e | 16,0% |
| traversa 1, c/f | 7,5% |
| traversa 1, a/h | **4,9%** |
| traversa 2 | 21,8% |
| traversa 3 | 11,0% |
| traversa 4 | 6,1% |
| traverse 5-8 | 9,7% |

Separare a/h da b/g spende un bucket intero sul 4,9% e lo taglia via dalla
cella adiacente che contiene il re arroccato.

### La mappa di questo run
E' un **refinement stretto** della mappa a 4: ogni confine che c'era e' ancora
li', i nuovi non fanno che suddividerne le celle.

```
        a/h  b/g  c/f  d/e
tr. 1    0    0    1    2
tr. 2    3    3    4    4
tr. 3    5    5    5    5
tr. 4    6    6    6    6
tr. 5-8  7    7    7    7
```

Quota per bucket: 27,9 / 7,5 / 16,0 / 11,2 / 10,6 / 11,0 / 6,1 / 9,7.

### Gia' verificato prima di spendere GPU
Sul branch, con una rete SINTETICA nel layout a 8 bucket:
- C++ e `sanity_deep.rs` danno lo **stesso identico cp su 71 posizioni**, che
  coprono tutti e 8 gli input bucket;
- 8 coppie specchiate su 8 danno valori identici;
- `nnue-selftest`: **41.577 posizioni, incremental == scratch** (quindi la Finny
  table raddoppiata e il refresh fra bucket funzionano).

### Costo a runtime, da misurare dopo
Payload 12.717.088 B invece di 6.425.632: il binario raddoppia. La Finny table
passa da ~34 a ~66 KiB per thread. Il costo di refresh sulle mosse di re era
stato misurato su 1500 partite reali a **1,20x, non 2x** (59,4% -> 71,3%),
perche' il flip speculare fa gia' scattare il refresh sull'attraversamento d/e.
Modesto, ma va misurato con bench6, non assunto.

In [ ]:
# --- helper: qualunque comando fallito ferma il notebook, e l'output si vede ---
import subprocess, os, sys, json

def sh(cmd):
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, executable='/bin/bash',
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait() != 0:
        raise RuntimeError(f'FALLITO (exit {p.returncode}): {cmd}')

sh('nvidia-smi --query-gpu=name,memory.total --format=csv')
sh('df -h /content | tail -1')
print('\nGPU presente. Se la riga sopra non mostra una T4, cambia runtime.')

In [ ]:
# --- Drive + configurazione ---
from google.colab import drive
drive.mount('/content/drive')

import glob
def find(name):
    hits = glob.glob(f'/content/drive/MyDrive/**/{name}', recursive=True)
    assert hits, f'{name} non trovato su Drive'
    return hits[0]

PARTS = {i: find(f'hydray_v7_part{i}.bin.zst') for i in (1, 2, 3, 4)}
# Taglia attesa DOPO la decompressione, per parte. Una decompressione
# interrotta a meta' produce un file piu' corto e nessun errore: senza questo
# controllo si addestrerebbe in silenzio su dati troncati.
RAW_SIZE = {1: 23_742_906_368, 2: 23_742_906_368,
            3: 23_742_906_368, 4: 23_745_983_264}
for i, p in PARTS.items():
    print(f'parte {i}: {os.path.getsize(p)/2**30:6.2f} GiB compressa  {p}')

NET_ID   = 'hydray-h8-v7-160sb'
TOTAL_SB = 160          # stesso budget, stessi dati e stessa architettura della
                        # rete adottata: l'unica variabile e' la mappa
STAGE    = 20           # otto tappe
ORDER    = [1, 2, 3, 4, 1, 2, 3, 4]   # ogni fetta girata due volte
TRAINER  = '/content/th/nnue/trainer'
assert len(ORDER) * STAGE == TOTAL_SB and STAGE % 10 == 0

In [ ]:
# --- Rust ---
sh("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal")
sh('$HOME/.cargo/bin/cargo --version')

In [ ]:
# --- clone + verifica architettura ---
# La verifica non e' cerimoniale: un run precedente ha addestrato per un'ora
# un'architettura diversa da quella creduta, perche' il clone era sbagliato e il
# nome del checkpoint non dice nulla sul contenuto. E il tentativo a 8 bucket
# del 2026-08-25 sarebbe partito con trainer_deep.rs e sanity_deep.rs ancora a
# 4 bucket: e' esattamente cio' che i controlli qui sotto adesso impediscono.
import re
BRANCH = 'halfka8-v3'
sh('rm -rf /content/th')
sh(f'git clone --depth 1 --branch {BRANCH} https://github.com/ThomasGhione/HydraY /content/th')

tr = open(f'{TRAINER}/src/bin/trainer_deep.rs').read()
assert 'const HIDDEN_SIZE: usize = 1024;' in tr, 'trainer_deep.rs non e a 1024'
assert 'const L1_SIZE: usize = 16;' in tr, 'il layer intermedio non e da 16'
assert 'const OUTPUT_BUCKETS: usize = 8;' in tr, 'gli output bucket devono restare 8'
sd = open(f'{TRAINER}/src/bin/sanity_deep.rs').read()
assert 'const HIDDEN: usize = 1024;' in sd, 'sanity_deep.rs non e a 1024'
assert 'const L1_SIZE: usize = 16;' in sd, 'sanity_deep.rs non ha il layer da 16'

# La mappa e' IL punto dell'esperimento: leggerla da tutte e tre le copie e
# pretendere che coincidano. Il tentativo precedente e' morto proprio qui.
def layout(text, pattern):
    m = re.search(pattern, text, re.S)
    assert m, 'BUCKET_LAYOUT non trovato'
    v = [int(x) for x in re.findall(r'\d+', m.group(1))]
    assert len(v) == 32, f'BUCKET_LAYOUT ha {len(v)} voci, non 32'
    return v

RUST_PAT = r'const BUCKET_LAYOUT: \[usize; 32\] = \[(.*?)\];'
CPP_PAT  = r'inline constexpr int BUCKET_LAYOUT\[32\] = \{(.*?)\};'
hpp = open('/content/th/nnue/network.hpp').read()
maps = {'trainer_deep.rs': layout(tr, RUST_PAT),
        'sanity_deep.rs':  layout(sd, RUST_PAT),
        'network.hpp':     layout(hpp, CPP_PAT)}
ref = maps['trainer_deep.rs']
for name, v in maps.items():
    assert v == ref, f'{name} ha una mappa DIVERSA dalle altre: {v}'
nb_ = max(ref) + 1
assert nb_ == 8, f'la mappa ha {nb_} bucket, non 8'
print(f'branch {BRANCH}, 1024 -> 16 -> 1, {nb_} king bucket, 8 output bucket: ok')
print('mappa (righe = traverse 1-8, colonne = a/h b/g c/f d/e):')
for r in range(8):
    print('   ', ref[r*4:(r+1)*4])

sh('apt-get -qq install -y zstd >/dev/null')
st = os.statvfs('/content'); free_gb = st.f_bavail * st.f_frsize / 2**30
print(f'liberi {free_gb:.1f} GiB, picco atteso ~36 GiB (una fetta + cache di Drive)')
assert free_gb > 45, 'disco insufficiente'

In [ ]:
# --- helper delle tappe (ESEGUIRE SEMPRE, anche in ripartenza) ---

def load_slice(n):
    """Scompatta la fetta n in /content/data.bin, sostituendo la precedente."""
    if os.path.exists('/content/data.bin'):
        os.remove('/content/data.bin')          # spazio prima, non dopo
    sh(f'zstd -d -T0 --long=27 -c "{PARTS[n]}" > /content/data.bin')
    got = os.path.getsize('/content/data.bin')
    assert got == RAW_SIZE[n], f'fetta {n} troncata: {got} != {RAW_SIZE[n]}'
    print(f'fetta {n}: {got//32/1e6:.1f}M posizioni, taglia verificata', flush=True)

def stage_cmd(end, start, resume_from):
    """⚠️ STAGE_END DEVE STARE ATTACCATO A `cargo`, non in testa alla riga.
    `STAGE_END=40 cd dir && cargo ...` assegna la variabile SOLO a `cd`: cargo
    la riceve vuota, il trainer ignora le tappe e tira dritto fino a TOTAL_SB
    senza salvare niente. E' costato un run intero. Da qui l'`env` esplicito."""
    args = f'/content/data.bin {TOTAL_SB} {NET_ID}'
    if resume_from is not None:
        args += f' {start} checkpoints/{NET_ID}-{resume_from}'
    return (f'cd {TRAINER} && PATH=$HOME/.cargo/bin:$PATH '
            f'env CUDA_PATH=/usr/local/cuda STAGE_END={end} '
            f'cargo run -r --bin trainer_deep --features cuda -- {args}')

def save_to_drive(end):
    """Copia il checkpoint su Drive e VERIFICA che ci sia arrivato davvero: il
    mount di Drive scrive attraverso una cache, quindi un upload mai completato
    passerebbe per riuscito."""
    ck  = f'{TRAINER}/checkpoints/{NET_ID}-{end}'
    dst = f'/content/drive/MyDrive/{NET_ID}-{end}'
    assert os.path.isdir(ck), f'checkpoint mancante in locale: {ck}'
    sh(f'rm -rf {dst} && cp -r {ck} /content/drive/MyDrive/')
    size = lambda p: sum(os.path.getsize(os.path.join(d, f))
                         for d, _, fs in os.walk(p) for f in fs)
    assert os.path.isdir(dst), f'la copia su Drive non esiste: {dst}'
    assert size(dst) == size(ck), f'copia su Drive incompleta: {size(dst)} != {size(ck)}'
    print(f'tappa fino al superbatch {end} su Drive ({size(dst)/2**20:.0f} MiB, verificata)', flush=True)

def run_stages(done=0):
    """Esegue le tappe da `done` in poi. done=0 parte da zero."""
    assert done % STAGE == 0, f'{done} non e un confine di tappa'
    prev = done if done else None
    for k in range(done // STAGE, len(ORDER)):
        end, start, sl = (k+1)*STAGE, k*STAGE + 1, ORDER[k]
        print(f'\n===== tappa {k+1}/{len(ORDER)}: superbatch {start}-{end}, fetta {sl} =====', flush=True)
        load_slice(sl)
        sh(stage_cmd(end, start, prev))
        save_to_drive(end)
        prev = end

In [ ]:
# --- training: otto tappe, fette 1-2-3-4-1-2-3-4 ---
# NON eseguire questa cella in una ripartenza: usa invece la cella in fondo.
run_stages(done=0)

In [ ]:
# --- verifica finale e salvataggio su Drive ---
final = f'{TRAINER}/checkpoints/{NET_ID}-{TOTAL_SB}/quantised.bin'
sz = os.path.getsize(final)
# 12.717.088 di payload, arrotondati a 64. La rete a 4 bucket ne fa 6.425.664:
# se esce quel numero, e' stato clonato il branch sbagliato.
assert 12717088 <= sz < 12717088 + 64, f'taglia {sz}: NON e la rete a 8 king bucket'
print('quantised.bin:', sz, 'byte - 8 king bucket x 1024 x 16 confermati\n')

sh(f'cd {TRAINER} && PATH=$HOME/.cargo/bin:$PATH cargo run -r --bin sanity_deep -- {final}')
sh(f'cp -r {TRAINER}/checkpoints/{NET_ID}-{TOTAL_SB} /content/drive/MyDrive/')
print('\n' + '='*70)
print('RIFERIMENTO - la rete adottata (v7, 4 bucket, deep16, 160 SB), in locale:')
print('  startpos            64      mediogioco ~24 pezzi   897')
print('  KQvK               958      KRPvKR                 126')
print('  donna in piu      1087      re attivi (finale)     109')
print('  entrambi arroccati 1087     re in seconda           31')
print()
print('La loss su v7 della rete adottata e 0,012676 (4M record, offset 0),')
print('misurata con nnue/tools/holdoutloss.cpp. Quella del candidato si misura')
print('allo STESSO modo e sulla STESSA finestra, quindi e confrontabile.')
print('⚠️ Ma una loss migliore NON implica Elo: gli 8 bucket del 2026-08-01')
print('avevano train loss MIGLIORE (0,013732 contro 0,013830) e giocavano')
print('PEGGIO di ~11 Elo. Il verdetto e lo SPRT, e basta.')
print('='*70)

In [ ]:
# --- RIPARTENZA (usare SOLO se la sessione e' morta a meta') ---
# Come si usa:
#   1. esegui le celle da "helper" fino a "helper delle tappe" compresa;
#   2. NON eseguire la cella del training;
#   3. metti RESUME = True e DONE = ultimo superbatch salvato su Drive.
# La fetta giusta viene ricavata da ORDER: non devi ricordarti dov'era.
#
# Con RESUME = False questa cella non fa niente, cosi' "Esegui tutte" e' sicuro
# (altrimenti, a run finito, ripartirebbe da DONE rifacendo ore di training).

RESUME = False
DONE   = 20      # ultimo superbatch salvato su Drive

if not RESUME:
    print('ripartenza disattivata (RESUME = False) - nessuna azione')
else:
    ck = f'/content/drive/MyDrive/{NET_ID}-{DONE}'
    assert os.path.isdir(ck), f'checkpoint non trovato su Drive: {ck}'
    os.makedirs(f'{TRAINER}/checkpoints', exist_ok=True)
    sh(f'cp -r {ck} {TRAINER}/checkpoints/')
    assert os.path.isdir(f'{TRAINER}/checkpoints/{NET_ID}-{DONE}')
    print(f'ripartenza dal superbatch {DONE} (prossima fetta: {ORDER[DONE//STAGE]})\n')
    run_stages(done=DONE)

## Come leggere il risultato

Lo SPRT e' testa a testa contro la rete adottata, a 4+0.04 e `threads=1`.
⚠️ Qui, a differenza degli esperimenti sui dati, i due lati **non possono
condividere il binario**: la mappa dei king bucket e' compilata dentro il
motore. Servono due build, e va verificato che l'unica differenza sia quella
(`git diff dev halfka8-v3 -- engine/ uci/ tt/ board/` deve essere vuoto).

Prima dello SPRT, due misure che costano minuti:
- `./chess nnue-selftest <net>` deve dare incremental == scratch;
- `script/engine_driver.sh bench6` sul binario a 8 bucket contro quello a 4,
  per sapere quanto NPS costa il refresh in piu'. Il conteggio nodi cambia
  (rete diversa), quindi serve il tempo, non i nodi.

**Se vince** - la mappa a 4 buttava via risoluzione, e la decisione del
2026-08-01 era un artefatto del budget piu' una mappa mal tagliata. Si adotta,
e ha senso guardare subito se a 320 superbatch sale ancora.

**Se pareggia** - a parita' di budget la risoluzione in piu' non paga, ma con il
doppio dei parametri pareggiare significa che a parametro sta imparando meglio.
Guarda la pendenza fra il checkpoint a 80 e quello a 160 SB: sono entrambi su
Drive apposta.

**Se perde con pendenza ripida fra 80 e 160** - sta ancora salendo e il budget
non basta. Il ramo giusto sono i 320 superbatch, non l'abbandono.

**Se perde con pendenza piatta** - allora e' la risposta vera, e vale anche per
la mappa giusta: la domanda si chiude per davvero. In quel caso la leva
successiva e' l'architettura (L2 da 16 a 32, o un terzo layer), non i bucket.

### Una cosa da non rifare
I sanity eval **non predicono l'Elo**. La rete a 80 superbatch aveva KQvK fermo
a 657 e sembrava a corto di dati; ha poi vinto di +29,4. Guardali per accorgerti
di un disastro - mirror rotto, valori assurdi, taglia sbagliata - non per
prevedere il risultato. Vale anche per la loss.